# SWOT–Sentinel-2 quality-gated comparison: Campos Gerais

Project lead: Lissandra Kruse Fuganti. **The primary good-quality sample is empty (n=0).** This notebook cannot estimate reliable cross-sensor agreement or agronomic accuracy for this date and fixed area. It processes actual satellite-derived subsets, not simulated data.

## Scope and provenance
SWOT Raster Version D: 16 June 2025; Sentinel-2 L2A: 17 June 2025. Time separation: 17.047 hours. A fixed context AOI near Capao da Onca, Brazil (WGS84 -50.07,-25.12,-50.02,-25.07), not surveyed crop plots. Sentinel 20 m MNDWI is aggregated onto the native SWOT 100 m grid; incomplete edge cells are excluded and at least 80% valid optical coverage required.

The bundled NetCDF is a **derived 55 x 51 cell spatial subset**, not the full source granule. Source filename and SHA-256 are retained. The optical GeoTIFF is derived from the corrected Sentinel pipeline (scale 0.0001; no duplicate BOA offset). A standalone Sentinel notebook is available at https://www.kaggle.com/code/lissandrakruse/campos-gerais-sentinel-2-water-indices?scriptVersionId=363624325 . Code was developed with AI assistance and requires human scientific review.

## Interpretation rules
- Quality 0 (good) is the primary analysis: no eligible cells, so statistics are null, not zero.
- Quality <=1 sensitivity has 495 paired cells, all suspect; exploratory differences are not accuracy scores.
- The source subset has 508 finite SWOT fractions. Missing values are never dry-land zeros. Quality 3 is bad; here these cells also lack water-fraction values.
- Retain negative SWOT estimates (39 in the paired sensitivity sample); do not clip them to zero.
- Optical threshold fractions and SWOT estimated water fraction are different estimands. Many pairs are zero-heavy; correlation and small mean differences do not establish agreement.
- No field validation, operational irrigation advice, regional generalization, or causal claim.

## Sources and reuse
Surface Water Ocean Topography (SWOT). 2025. *SWOT Level 2 Water Mask Raster Image Data Product, Version D*. PO.DAAC. https://doi.org/10.5067/SWOT-RASTER-D (accessed 2026-10-29). PO.DAAC openly shares data without restriction: https://podaac.jpl.nasa.gov/CitingPODAAC .

Contains modified Copernicus Sentinel data [2025]. ESA/Copernicus Sentinel-2 L2A via Element84 Earth Search. Data terms https://cds.climate.copernicus.eu/licences/ec-sentinel . The notebook code license does not replace source-data terms.

This self-contained notebook reconstructs small public derivatives and runs the comparison and ten geometry/missingness/reproducibility checks. No credentials or private storage links are embedded. Internet access is used only to install the pinned netCDF4 dependency if absent; scientific input data are embedded. Outputs are regenerated, not copied from a prior analysis.


In [ ]:
from pathlib import Path
import base64,gzip,hashlib,subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','netCDF4==1.7.5.1.post1'],check=True)
ROOT=Path("/kaggle/working/joint-pilot");ROOT.mkdir(exist_ok=True)
p=ROOT/'SENTINEL_ATTRIBUTION.md';p.parent.mkdir(parents=True,exist_ok=True)
b=gzip.decompress(base64.b64decode('H4sIANi942oC/3VT3W/aMBB/z19x0l42VBKalTJ4o/RDrdCYCtUepok69hG8JXbms6nYX79zAnSdtLfY8f3u93H3DpZovDZY9XNQwgsQ3jtdBK+tSZJeb2aNF9oQ1FbpjUYFM9ugM1oGOtV2ld/yQT78nvZ6SbK0wUkEXYsSJ3+1uII57uLHFJ6X+dU6zx9uVutYN7g8H60H63k+fT4DWxC6Hfc6H8FDMAjxBTOD84+TfDzJB/C0mqUwq2xQfdt4Xevf/PoO7er+9hakNTt0xApAGAXL1XQGQkokAr91NpRbuKmwZlbw6QJuhPNb5iic3EIG069LWDRo4Jo1pfCI0jrFdcgQPgjWal9MZYWKolmiaf81zkZ8bUqohdEbJA/vJSNUtuQ31KD0zDAfw0J6W6CLki4/pEnShy+hqLSE2eKu9ZHQg0KSTjcxhLMj9ShFai/i5QS23jc0yTKHpSbv9imnYmJ5Kl4oo5Pj/SoXfWlLyrjTf6Pj/mg4sAwc/graMVVjPV/SayepKJUVR+oxlSegFEN2qKYMZf/YmbstNhstNTt26lVhyccO+RX4WEL/wCorQwyJsvxiNB5cZJfjwWg4zI5w65jQeh4x159bzCRZcRZvFDFgrT2xLo5IBdlZqqJnhylng5VoOl87k21daNOdKRQ/ODrwFiIKT5bS8UcXB3VjfrAqhXsPpd4h8Q28COeE8XuwG0bh3Apd6Xh0MdDghNyncG1jMZOrRMH++K12qt/wQO4POyAOzYjrdMmsKqhRUHDt+FKLRsQ2xVp6K100TaWZDHMPJraIE3gy9YzFKIwAjXU8pxEHPYuYxesWox1nRhVAyKTiuHP6Ht2JuDayCiquhrSB10ny5MR0RNVSI12atudhY54e5xQ3pukmnhGwsPYnpckfZzirdYgEAAA='));assert hashlib.sha256(b).hexdigest()=='a8cddf22d8f16e4d22906e3042fcd04ac92800b9848f88a94778e3e407e1c24d';p.write_bytes(b)
p=ROOT/'test_comparison.py';p.parent.mkdir(parents=True,exist_ok=True)
b=gzip.decompress(base64.b64decode('H4sIANi942oC/21W227bOBB911dw/bAUa1qRvVHaytUCiy36miLdN68hMBJlcSuRDElbdoP8+w51cQwngGGTwzPkzJmbZ7PZA9dGlftCPIpGuNPNjquWO3NCRc2LnxYpiVwtLKrEkZdIi0a5NZIKWcecsE4UrEEH1ogStoAtGibaaDabBaLVyjj0n1WS1szWjXikhT0ElVEt0sx5ARpB32E7Kch9q0+IWSQ1Ncw6boQalCR3f3/9djspfWWOWe6Ch8yrh3leiYbnOYk0M1y69X32cIML1cJWgBF43WbemKhRrAyV5jK8vzzPvduisJEHYUKCklfIcMBKkgaoE65Gkz3RqB7KOY6cqADuLbap4W5vJLJRr7gk1EbOMGkrZVpYF8YGXUUdLbIegG2nXN4xuDSvDCs8hZisn2hO87cIcIvlT3vm4wSoQh0uceCykLzJ+2DkcMgN23HAKf0erJVlJ94+XfLGsQs8P+pGGeaUOeWlqCoO1BZwa8Cs5RCEInIq59ruQpJlf6w+rlaIyRK5iGXZMo7HDc+yBewmpa6KbM00SMMkocmS9DCpI2ErIYXjYVeRyO5bf2kSf5r0wvApy2Ly+xXyDI2Dltmf2eVx78/ZWn88gW8/J9OzrGlCIGzjj7d/ZtEncnnyNMjBn/M9w0nRKDu+MEBoVw2LhdKjBJhrsiVffLzUFXZQhWXLmQzZo728hhDabnykrHDiAMGegp43fIm3G9wy/hqz7ZVVIeT25pICpQk4BbS9d/AFvLq8wJMzefEl9jH9HPSpP1ZbCDUlpN67mz4x4yT5lq/iVRLfLe8iWeC+DkqoF3HMhHTexa6GpAnLDT6C7Snw6KJinsRkE2/hQ9bi9AZ5OiOrxSsyQBKazIH7ALcsgnJveOnhr1kMeuK0SG5TcZovqTim4jhPllsKGpJJsknTxZKm2wC9E0gozOF+yj3fOShk/5g9JwMBU8fQTBgOBcabBgraHjDF3eB25dtEBqKoM8CvCSuy7sa16sINhm9AF6qBbw6tRMhd3sJaQjurp807tU6x0n2nHYv7HHv6bgN5hbvacFur5q1KK+Te5iPO5xCCFoXAQArmISHRL6HDD+eY+Hwgvg2iS4d82GBBqF+AHqE+tH41jxLyASqe+gh6zFngoU+b8aEtoRW0Y9dX37UM0vBaBAl8LRrq5iwFT/wYsNmzj3jL02f8eHLc4hRyG0ZWCM3C5Vb84kBEzVbJHU7H6RQN+1D3zTvv1UJoLjU/lmLHLei+9Cxpz8995FkohQkJEhXylVPBjPRHm2EmUBxpufM/Q5YMg2Xbt5bBuN8yrPYOiilvmRQVPDFgXgKfaO8ekYH+3PGjC/txVsK8tOEzHuZ1rn1mlzhdxhQzA3ZA3MH7npQXYL+EAZCtyBz/C5cF2vh44O9//fiRomWMpulPUSushZSU3FqKCrWXrjd8GJLjn4M1ui6GHuP55BZ5Qx33r/wPIQQJKmcIAAA='));assert hashlib.sha256(b).hexdigest()=='763d8ab3ac325ee7a5d2dd9b50cd466a270378294b16207c7acba6c49b5aa52f';p.write_bytes(b)
p=ROOT/'SWOT_ATTRIBUTION.md';p.parent.mkdir(parents=True,exist_ok=True)
b=gzip.decompress(base64.b64decode('H4sIANi942oC/31STY+bMBC98yvesZWCSZBI1e0JwVa7UhoiSLvHaNZMiFuwkW2i7r+vnVTdW2+WNW/e11RGe1LageBm8opGuOXVsUfPVl25x9maCd1Lc8SOrzwixwt5tvhG7hdacvH9PNHAqMkTDtb0i/Qr/GDrlNGoBbrFnknyX2AjmTSOZjaDpfnyhg9x+8cV8nVeCOzLrsShEXVZVgJ18/yAi/eze8iy3ihh7JBt1qJYbz9lEZe2ZXd8bNMaAqWU7FzQHDZt0806zT8HcrPYwN1HccGoV37xUVe1f+y+3Niy+LxbnJSLokWSNDNruAtZpQfMZlTy7V3IbHoiKX7Oo9DkSAzmmlXKh9FDE3UHLTGMEOP/Mf196GbktMtPT+3pnugp2EmS40U5qNhNUfwuNqnkcbz3Epq68gr7IJkgzTSP7BnGqkHpUGEIVi8j/zN/ViNrmhik+/ep7qnMiy3IMizHKwjJKY09+6r+CvLeqtfFswudGMw31lByvAp8b3crOJYBF/bBqSGCw2+Uq7Qcl557kfwBulokIV4CAAA='));assert hashlib.sha256(b).hexdigest()=='56279b4c6326624a4c374b5613e722cd5eaaeb70ce97486b794e21f268c2144a';p.write_bytes(b)
p=ROOT/'README.md';p.parent.mkdir(parents=True,exist_ok=True)
b=gzip.decompress(base64.b64decode('H4sIANi942oC/02QQW/bMAyF7/sVvG817CDKMuRUrN1QDE2BdUCPASMxNgFZ8kjKi//95LUodiUf3/v4fpYE02JDTuDzOKHQCb0VjM20gA30vjVSO71KWHNa13gxEuCkhjFy6kHod2GhkZLp/1q7WgMPaSoGR7Kvd9+AFRACCc8UnLu67sZTjKDlrGSfIGWDS4mxF0wlUgNPwj0njJqLeBpQh5oLbAqvk1N+E5x0wI3bAZoJn4vV2+eKw4livVgJarTPIuSNwiR55opxox4jBXg83r08HEDwzxlT0I9CVRGKN64N1MBJOMsP7PtI86aB++v0ZsMjypLaAyglZeOZbUnbL+4Aj7f3bdN2++3GuV3bVfO2ca7r2v3n7W57gAsnNnp+efrl2n0D3wWXkVVrnbWEIEusINWW+0RhVY04oS7jSPU/D/+4GzhmwF5yyiP7GSMHXInX1+m9iTrQ5sNfBgm+5PABAAA='));assert hashlib.sha256(b).hexdigest()=='b54f4d9f29a33cea69ae67db4051d845809b3c72bc5c3c6b407872766f68f7c1';p.write_bytes(b)
p=ROOT/'requirements_comparison.txt';p.parent.mkdir(parents=True,exist_ok=True)
b=gzip.decompress(base64.b64decode('H4sIANi942oC/8srzS2otLU10jPVM+HKSy1xdnEzsbU11DMHChjqFeQXlxhyFSUWl6QWZeaDxE31jLhyE0sKcvJLcjKTbG2N9QwN9Cy4AFxgLfpHAAAA'));assert hashlib.sha256(b).hexdigest()=='38a1d5b1edc6840b3ff59a4592f9829567bd16a718e260f60d34e87b7a597bcd';p.write_bytes(b)
p=ROOT/'compare_actual.py';p.parent.mkdir(parents=True,exist_ok=True)
b=gzip.decompress(base64.b64decode('H4sIANi942oC/5VZ0Y7bOLJ9768gskAoJbLadtLJjD0KkJlMBrPYSYLpDPLQawi0RNuclkRFpLrtDgLsP9wvuLgP+7RPi/sF+ZP9kj1FSrLd6exmJpNuiawii8WqU6eUe/fuPc9sKwpWCauu5GjdqJydv3v99vRcVlZVsmCZLmvRKKOrOas0E63d0FQGBV2xRr5vVSNzttINK3SGpVRVt9bE9+7dO1FlrRvLtDnRJpbVlWp0FRtpc7kSbWED/subv/zw+tXLn3968fOvPOKntqxPy7pI/aaShyerRpesFnZTqCXr1nuD137t32FYtBGG5vuxqi3rHROGVbXXr6T94cXLx73+C2EFrOjFG2GsbJT2sv1bfC2autewjagMjlimS91WuYkaWTf6d5nZ6FdpBExW1fqW/qDTL0LTqW7UWlX91qWwdaEt2b5/jFsjA/58vcbxP5OL6x090enqwp78+vr124T8EaTpShUyTcOYPFfZOXv929uE5k/5/g75HKNxeZmrJpBbZWyqL5O3TSvDk/NpJ30p1muspFtLF5leTU9NFwyn59Pv0+n0zz++Tafj6dn4yeRpOk7/Mn3OT87fddru+k/Ntbbp+OzsZS/4JK4yfoKLZ2YjgjqcNdK2TcW6q4sxOj17EtRxI0WeLndWmiAM443c5motjQ1CrwzXiiLYhrMTplZMGVUZK6pMBtuoquMqF00jdsPq29jqQjntu8TXssJ62YG4srIk4X7g5FrZTR8wwfm7kByfY/NtAn1h3HZBfsG3fHExW4Tz3a3xXTe+TG7HUMB/fHP+0+zxo+kThL57fjR9Op3yaHQ2jsdPo9H0LJ5M/dvUvY2f0jnc1tcb2cgg2D5LlhfjRXg/2H6Hp+kiDPE6V7sDmR3JTEhmRzKPOpmtSbYXaruY70yyu1A72DkbTRYnzG6Tg1ANtgbSMCLa0cNDPEzG7m84x6XVMgkKWQU7E0b0e2tC2EhXRRcZXA6+hT2liBGjhczhmMsFtsR6M/zErg8nkdq6161/XcCJdlcjEVaFFvbJYx66CxZV6OyMZrD0epW4Xfi1QM6lq0ZkPJy/B6AdjSMfREqjmGyr7DOdFIOysZhdKmvuVk0xda0MQRJbFWJtkg/VTFU2gFGmLYOAVC/wosxKVQij4HoV7g/RQvTRlIf3y/DZOAwdXFYRoKFiN6qmQPnSfouYtktLKSpgjIkNsAbxHH2FijCXJvx4woS1Dey9nHXZc+X3v4yuaP88TtNcZTZNXfQj7ShVKmDMZYxkaayhHAj4tpErjuVcRgwop2tc+vn0lJdVfq1iqyBDOWIoQatcbhPjUjqYhPMrUag8OfSRk0AcNVlqSXLIETeUwWoT4+fcLPHg08YHFyAqEAQCLDeWlly1RRG4eOzCJMrJ84kPH3L9fMDsQBwHF81GWCdydvQmJN6qaLDE/ybBAyEI0AAJHOUwDVY6B3AknT3NUCiSfc2IxZVsxFqSdS5NoHeCmutHEzqm8xqi7k/sx21WtLlEOWws7pHJfC1ZJovCsKXMBMoG88thYSa9ME3tdJWPjG6bDLpqKwsTI4jhsISS+5WuJLIJGf4sMcu4kCtLcDJMIOO/owmgwYZmAAOziOYGlaW2VpdHU72S1XV4cJ4BlGj3qB/uExtGIRuJVyBajW0CG84wlasrlctg8MV9HzbPbBjul0C1+rowiJwBSa+IbHTJYCkVLkbxJBrHUTxZIG10TTSnSAajLjiQmC/mZUucKbmV7PcP3ztd+GTYKIm/wQlxLORa4pe4HzisSghKW1PLWxPfJZMOS0tJagFldISlXeCL5Hp1QSOL+U2Cwe45lwViToxuINLF1AdecYdVJOHQCp7jpZBpfzQ+c/4hMCOgCcTSBG6hMHRoQMguQobIkcyFBScxwMZqBWcCPVNX8EtVtSbtzv7FxbuF7163Kc2dZpn3za0VHjyYftG6WooGbCdtDlbIdNNkWq4CEd2gAEaoiHvlZ1MmqpyKlLE5vY8P32/o/WB5f1jywG1LhTfwS3b1rrlT9+brdCtd3chGpy7x+WENuqHS0llXybWj9HeIie8g9pE4DnHL5AMHytsWMly4ZgD8xG5SjxcmBWbil5F56kMXA6oUzQ52pLIAQ1gW/SYRF1qly6Xeptdr881jPrv4D1RmEXFqN7Ctbz5c38HAK1jJ9kg67wDOyALpgUZjucNIheJjkLEGwLAn55h9/vpn5uoEQWADDVGBy8Iyhwlpo68NmgtyiAcJZ0JHyNLWltMppi7sNs4i/FiNnBR4yQPiPDT80I9MhpHVovO4I76p57FueaKLR3OO1fLZ+Tuqq6jh+JUaddMHFJVptWydjCvZGO+IN2p5pVbS3Fqfym43EVMjBHTjKOG4VtyikagTrktLN7hLLDp5Go8fP51883j4r9vZg1a6Z0RO6zhyDqFuwBCnTVCl7A5+bSv0C6IoelUH4qT/PgzdOh3eve9XcND7nqAXO7SVet9KJ3JEpGggXIQfI97v5AhYSsD6n013cvvM6aE4Xctvxiny2/WZhbQ+gP0BjlLlCLxx3j72O0PGfNYhs8+NqId9d3NGIazJ3N7sQk72Ch7vDzTsBgxhowsk2pd0P9jP1LvqFWly4lCpeiYHn93yjKe6Amfcpd0itAfCrECyn6Vn4/TbM05ltx8NoHLRiR7VOIyHi+giHp9FMf7/9mwRDu1WxAtVKuvCjzKKv9JsrXU+6s7DnFmj3iyf5LOuOrLOzQffHlgmKqKkS8kQ7JgFFMRI63Nv10hXxY7ty5FhYMWgQGiVkQKalmq0MSPyLNyVwSiYCcld5JiuyLIWpuwYJsW60ZUuVcYc23BHoK1+efXiHcBFVWTZcFmDz9FcuqXogwgUkbjMao9pt87aHUA3tOrzGgC7dQcqdpOnG9bnLyuVwWi2cZ9diPGAaee0LKzvhCqpW8NgL+ALMNjaDS35qoN+v/kQE8QGcO8y90fOClXXMp/TPobQ0knTISRiHhvC5hoqAFXQeao4/hy0w0uQyNzbsbWEuX5J0zZXckczja4ZfaYwzvhaGzva6IytpV43ot7At7gboLZzLXGt4PVvR98pUh/npoO1+LqhLKftAhqJ87asTeArWKScX5Jp1H1XSnyfEz7kf62oX4OH6dsIKl3ewC8gBfynt4gVnGQjidV21QA4j3xRud30AxMMdKgwibijklDuuSTmCFaPeD8fqhGfoTfgvgdwCUVs1B0RoUNqsLaARzmO79pBUcpIUBp/8Lh6nLl8dr2KDieGvg/pxGf0EN2ld5jxfIaXg6riAnyART4b2PRB4aG27jNTOtSK+EGKHbBBd1xP9Hugul6NeqWO7X/sQYrI7B1NJUVEQE55yH1bibvh0YMH3W36LzHGzvDXh8edbR0xaEQNxYcr66co2+Mo+KyUh/uJ46rfTdC4e1rgttQ6EluT1IVFlVi6QA+m0TTCBNXzJJhMom/DqBA7aks4wQ3CgnLPfT4gkNlGIrLKFjK6AmvGD4xkpahdGxII+ugSIRwHP3b9APzY+w/gN4DLkOWAUO9wppfIgisPwIzgkUfjKH5yFvHvixYUI4y6XSYHuxzWu+jWfWG/LihGU+bB8Bkb7/EvcNQtvHubydFhvhwUEf/xALT3ETXz8OQajL4x49GINnKb/Zp/3/JwQaGUle5i1hJdORwa+Dni1M7fR9807vAg70g36YZoGupdEM6zkj5bp0v6MPSnZU5/eDjHZu6T78HXWXBLTczUf7Z9e63PCwTzK8ABhGE3GltAAoHD4VxAMZCM4jFK6ZVnt8nYxUSC04Uf/7DxHzityGcutjgtRM9iC6iBbYnYxqpE+bpGL9Qlh//lQjDJSqQZGYuDLxMEtT/WUoDMlYh86Edm06jqkmgRueHr7ZtlS+dK3OClCS78mRExkyh+5GjEE9CIeSdUiKUssJBaV3JfaefI/F3pCwRzljGDcHCf5pBY3fKwJvAJtkKdcmk5QZ/dzW+7pX97+8t0es4kgINqYHCJAB6Edp0Qrshu9tMnzLmAgnqycLIZ6Lps+qa/J0sLxPf+2STfRqKoNyKJH7sdKGSCC5csYFHdb345GvGouEbnvzdVlcGI+iaKhb1td432x3rtLdnTlNHQP+2v4vYx3X0N7Oqzm5sfuXZ1TL0OqVYtVAMuVyUfusN7sv/RYR+CybS1X4P/IOpP/6dZLtjr6tPfxcAoagXfzNjkyb/+9j+Tp+zPbSUZ/TvCX6tzTxCPqKT7liax33jOfmrEjiWsrcSVUDhXIU+7D2Ed7clhYAHqzw/CYtobJq4kfn/GRcBU6JtZXa15lNcqeUTfvk98xJvkgr9sxKe/u4NI9ul/161wYAUo1iXlcg/FkPl/MFPnFFBj6pcHzU//dDd2gKsDnPIXyoEgOah386d/guD0mCgrbboFNI7F34AZmCNRPeyq77yYxVCU3Jn6r9KIcfqSbP2oCWf/Jbn6VVwNG0rYUGX6OrCY3Z2E2ALO+nIKyoP888l3tMpnvnQhj9AViK9aFppRSyIat/iB/u62/nCLPhd6bXL25yH8hkJVE8vFVeumcuq34nofyZj7va02TsgH9PNSEzeglqfKVI2cxdRSC+aiOxd5F9iZqm4EIhtcV5laV5/+cSULH9uf/pFrBDftaICGOLeLgz8Y3mltb0U4VdGs0AbJrtaORlNnfEC/P1zOPAO/uFz4f1FwN//fvyn84U7+js77yy3217TSi49D1xCGJ/8G5LCd2AgfAAA='));assert hashlib.sha256(b).hexdigest()=='4d781b72fff43352a5b60b6d69953a796d9c035dcfe54d62d4e09a87472e2051';p.write_bytes(b)
p=ROOT/'input/swot_055F_20250616.nc';p.parent.mkdir(parents=True,exist_ok=True)
b=gzip.decompress(base64.b64decode('...'));assert hashlib.sha256(b).hexdigest()=='9b0784947d429aa3f6a7b9f8f3a0e3b8aa11d4e5ab76e6ec1c3990efd89e7cad';p.write_bytes(b)
p=ROOT/'kaggle_outputs_v2/sentinel/S2B_22JET_20250617_0_L2A/mndwi.tif';p.parent.mkdir(parents=True,exist_ok=True)
b=gzip.decompress(base64.b64decode('...'));assert hashlib.sha256(b).hexdigest()=='18ca1ad5a1e3d9e6258a92aa90a734a16e220f578bd4194411a18d70f67d2914';p.write_bytes(b)
p=ROOT/'kaggle_outputs_v2/sentinel/S2B_22JET_20250617_0_L2A/manifest.json';p.parent.mkdir(parents=True,exist_ok=True)
b=gzip.decompress(base64.b64decode('H4sIANi942oC/9VYa4/cthX97l9BzNfMg5dv7sIFHNcoHKQN0DUQIIYhUCS1o0YjTUVp10ng/95LzVPrpkC7LWoPFguJl7w8vIf3pd9eELKoh7gr6rC4IYs79m3B2Hdv3hWMMkkV6IIW37NXi2WeGNwQh3oX88wsX1G1Av0O+A2zN4yuOaP4++kw2XV1UZbdx+LxPhmBS97jKCErSddULw/PTK6BLS/j7DJONT5+mDR147Afh6LsxjakYhx2jKWzPmm5FvSwTjMGgh5fpDXmSqCsoWeN930dij6mrhmHumuLHapj9AA7pThk9b9NK3FqjO35FQdSN/Z+MsF2GPbpZrNJsR3qNjYr392ndeLrMa0eYxpWbO127teudY9p7bvdZWJiq4a5af6Gsc13mzfvNtmgG7X5PQY231K+HupqsZwDeXqM90cxIYAGXc7eji8fLiq8a/JRUEYpnIdjVUU/1A+x6KoKzZFnnIUh+sb1MRRpcP4yYUXXFwW+qfdF2jomVTYUuJJDSSlXIWqIVvGgbZRgfWW8i0FoVnrNZRVKpThCEUxHHwGQTwqaLya9nw7qF23dfxl8mK+WD7QpY5X2FEpjtCm9EAxKJMmXwqIXIxOycl4LKD0SpyvuaET6XKCMaj/nIz3WPagvghKAf48SNqOE/R8podJp4aCKilo0dwxcCImOIgwLVPqKeu8tWCcqZSpVGZylog6KA6uYiuwJJb75Evi4e/39/4yP51FBf48HZZSwxkpprQPvjFUlMEwf0pQakIIYKuoCZhwIvuRRC8ktdWCj0QpA0TkPuPGXEarE1xqqbMWRi0CNKUsWTBTKRGNtjF6zyjjvS3CS0Wio0xjUPGh0mCpa8PifUjPno2zG+GUQwr5WQipryiC9KiWN1mtrQEhvefTMltbLCvNFqHRVujJYqZxmSlQRBOfGmMoYeyTkxZGUxS4O2+7iJegyVYOAXXvg5Y3rh+1ddL3fkq3rd11b/xoDef3Dn25I2bkj5sLt900dw8uhHyNJHTmfmhwmvKRLMhno5cE8t8R3bdXUOKe9JyczHCeTPg4OuQ2k6nrixlAPt+QBSBr3sU8xoFOf/Tu5He7c3k9ple7IFXziHmLv7lHN0GEs293mf+Tux7d/JS0eCC/VQX9T37c7vE23BOPlSXTaYefSz1NhjiKxlEtFurb55ZbEj74ZQ0wEmQnd48Y33Rg2qcXHscXTpFRXaJBNcP3Pm7YbsHR3J50nsXf5pmXtf/7LH398+wdK6qYZ09C7yXJj+4DYcs2PJ9giqG3XhFuCysh9n2txgtYetoszlWi2vo4PeJ3GwR97BGwQ6IrZdwxumL7BElYKoUF/Q+kNpYdG4e/uwn7p2lCglX32riYm9BdZSFpYWQDOv3LdU11+8ZHMrFRMWGu11EYq4LC8lnKqstCCVNJiialmUiVB5vYFpLCaUTMTGkt1Xkst9hdWXMs4A5218rwehJKfeeahYJ0BBQ0TFM0xaqGf8GuNYCimH2s0hgyjlJjhZIrZjJMD2pFJNsMiFJ2wIHwhKWfXMsVMluExwHAlP4d5ruPmSIWhhx8SR7EemSEFi3/5GJghsROzM6SYODMY5AFzo6UzNBnMpNUAVYyDnh0D2HFLrQzXoD7Dekisc+5RiTjsl000px4wp0/2ZsxwbAtnWKg0ero2eCe0lWK+1GorRF4pgWIUk3PueWbKCjBYISilPwN6zDhPbunhMmlFkQk+J58CVuRZClyiWZWcX2E0YwYjMZ8oiSd5cg6u892wkistOKg5VrwUeKsEtsV4ieAEdZYg6zbEj8Xk94VHH7/qhPNFDo81vmvAmH8+4e56dKZs6AbXFPv6Y2zSfNnisAOWqf9KXPXOHyMUnNJfLm3/CbL8fUEJAZfUJvOIZucbucg3W83wTciLfZfqKT+ekZx86ukEDMyu2Lopx9qjbRePrm+P8f8VykmdMDbvmw6DaNf/cgmcqxRzYsBYmtWQDnMDeURBJK9+eLucoqrvu/0q7aPPsZlgarjE36rpuoDbrC/RNkf0wg0YcsvxFMdfd23OXInscHaO/+R1h9G0rf2YyN2xYiF5JXmfS5IPhwiMibEu+ykbYAlSxT6286rovh62YznVPW+amJOVEZuYUzMeK+fmTagT7pFQA1ZBanFK7W5Cecq1/6nyOqUxpo2G45elpimO34S2Lm3j1deaibGptMr52HgHLkgHkQcbFbqos8w5LNg1x2YL2yZGK0wVZRCATg7gwARNK6UDsyAWlw8OZ5XP7ZyXV1+Vzkqf+3lk1oiftD67ozyd/sqiaM6qLClzVJYes5SDQL30YCoJ3gbtDQTBjefWg1estLzUDvsktILKJllcnPik8tkN10llrmDzl8z139LBGSDzbCE4NJp1nnJvcsQFwXDESO6ZqTRDRrFqqIwukXKjghCO4yj11Vnz0W+KfR8f6vi43h/cXVTBexkqV0rhMJBTZE0yiIbTMuBmhmkfIz7bUhis/a3H8MxMvoeOagmLq6bkZI1nV9eXxvOk8r/RQX168ekfx6r9GSoWAAA='));assert hashlib.sha256(b).hexdigest()=='e92deec6f63afa41f79ee39793b12bf0be4364d10390661e7b3bb6628ced0b2f';p.write_bytes(b)
subprocess.run([sys.executable,str(ROOT/"compare_actual.py")],check=True)
subprocess.run([sys.executable,str(ROOT/"test_comparison.py")],check=True)


In [ ]:
import json
from IPython.display import display, Image, Markdown
report=json.loads((ROOT/'comparison/comparison_metrics.json').read_text())
assert report['primary_quality0']['n']==0
assert report['primary_quality0']['mae_fraction'] is None
assert report['sensitivity_quality_le1']['n']==495
print('PRIMARY RESULT: no eligible good-quality pairs; reliable agreement is not estimable.')
print(json.dumps({k:report[k] for k in ['primary_quality0','sensitivity_quality_le1','threshold_sensitivity_quality_le1']},indent=2))
display(Image(filename=str(ROOT/'comparison/comparison_panels.png')))
display(Markdown('Gray = unavailable/excluded, NOT dry land. SWOT panel uses an explicitly signed asymmetric scale. Suspect-only plots are exploratory, not validation.'))
